In [0]:
# L'obiettivo è capire quanto producono Nord, Centro, Sud e Isole
# e osservare come cambia la composizione interna del mix rinnovabile
# tra Solare, Eolico e Idrico.
#
# Il notebook usa la tabella Gold finale:
# progetto_meteo.gold.dati_studio
#
# Questa tabella contiene già il dataset finale del progetto,
# con dati meteo ed energia unificati per Area, Data e Ora.
import pandas as pd
import plotly.graph_objects as go


# Imposto il catalogo e la tabella Gold finale.
catalogo = "progetto_meteo"
tabella_dati_studio = f"{catalogo}.gold.dati_studio"

colore_solare = "#FFC000"
colore_eolico = "#40DD3B"
colore_idrico = "#00B0F0"
colore_media = "#FF0000"


# Seleziono il catalogo del progetto.
spark.sql(f"USE CATALOG {catalogo}")


# Controllo che la Gold finale sia popolata.
# Se la tabella è vuota, i casi studio non possono essere eseguiti.
righe_gold = spark.table(tabella_dati_studio).count()

if righe_gold == 0:
    raise Exception(
        f"La tabella {tabella_dati_studio} è vuota. "
        "Prima esegui Gold_Dati_Aggregati e Gold_Dati_Studio."
    )

print(f"Tabella pronta: {tabella_dati_studio}")
print(f"Righe disponibili: {righe_gold}")


# Leggo una query direttamente da Databricks.
# Dentro Databricks il notebook può usare Spark senza credenziali locali.
def leggi_databricks(query):

    return spark.sql(query).toPandas()


# Mostro i grafici Plotly dentro Databricks.
# displayHTML rende il grafico più stabile nei notebook Databricks.
def mostra_grafico(fig):

    displayHTML(
        fig.to_html(
            include_plotlyjs=True,
            full_html=False,
            config={"responsive": True}
        )
    )


# Converto colonne pandas in numerico quando serve.
# Questo evita problemi di visualizzazione con Plotly.
def converti_colonne_numeriche(df, colonne):

    for colonna in colonne:
        if colonna in df.columns:
            df[colonna] = pd.to_numeric(df[colonna], errors="coerce")

    return df

In [0]:
# Studio 2A - Confronto tra macroaree italiane
#
# Questa analisi confronta Nord, Centro, Sud e Isole
# sulla produzione rinnovabile annua totale.
#
# Il grafico mostra quanto contribuisce ogni macroarea
# e divide la produzione nelle tre fonti finali del progetto:
# - Solare
# - Eolico
# - Idrico
#
# L'obiettivo è capire se la produzione rinnovabile è distribuita in modo equilibrato
# oppure se alcune macroaree hanno un peso molto più forte rispetto alle altre.
#
# Possibile utilizzo:
# questa vista può essere utile per analisi territoriali,
# valutazioni di capacità produttiva e confronto tra aree geografiche.
#
# In ambito economico può aiutare a individuare macroaree più centrali
# nella produzione rinnovabile nazionale e aree in cui il contributo risulta più limitato.


# Imposto l'anno da analizzare.
anno_studio = 2025


# Leggo la produzione annua per macroarea dalla Gold finale.
query_macroaree = f"""
SELECT
    Area,
    CAST(ROUND(SUM(Solare) / 1000000, 2) AS DOUBLE) AS Solare,
    CAST(ROUND(SUM(Eolico) / 1000000, 2) AS DOUBLE) AS Eolico,
    CAST(ROUND(SUM(Idrico) / 1000000, 2) AS DOUBLE) AS Idrico,
    CAST(
        ROUND(
            (
                SUM(Solare) +
                SUM(Eolico) +
                SUM(Idrico)
            ) / 1000000,
            2
        ) AS DOUBLE
    ) AS Totale_Rinnovabile
FROM {tabella_dati_studio}
WHERE year(to_date(Data, 'dd/MM/yyyy')) = {anno_studio}
GROUP BY Area
ORDER BY Totale_Rinnovabile DESC
"""

df_macroaree = leggi_databricks(query_macroaree)


# Converto le colonne numeriche per evitare problemi con Plotly.
df_macroaree = converti_colonne_numeriche(
    df_macroaree,
    [
        "Solare",
        "Eolico",
        "Idrico",
        "Totale_Rinnovabile"
    ]
)


# Controllo i dati letti.
display(df_macroaree)

print("Righe lette:", len(df_macroaree))


# Calcolo la quota percentuale di ogni area sul totale nazionale.
totale_nazionale = df_macroaree["Totale_Rinnovabile"].sum()

df_macroaree["Quota_Nazionale"] = (
    df_macroaree["Totale_Rinnovabile"] / totale_nazionale * 100
).round(1)


# Creo il testo da mostrare sopra ogni barra.
df_macroaree["Etichetta"] = (
    df_macroaree["Totale_Rinnovabile"].round(1).astype(str)
    + " TWh<br>"
    + df_macroaree["Quota_Nazionale"].astype(str)
    + "%"
)


# Creo il grafico stacked per macroarea.
fig = go.Figure()


# Aggiungo il solare.
fig.add_trace(
    go.Bar(
        x=df_macroaree["Area"],
        y=df_macroaree["Solare"],
        name="Solare",
        marker_color=colore_solare,
        hovertemplate="Area: %{x}<br>Solare: %{y:.2f} TWh<extra></extra>"
    )
)


# Aggiungo l'eolico.
fig.add_trace(
    go.Bar(
        x=df_macroaree["Area"],
        y=df_macroaree["Eolico"],
        name="Eolico",
        marker_color=colore_eolico,
        hovertemplate="Area: %{x}<br>Eolico: %{y:.2f} TWh<extra></extra>"
    )
)


# Aggiungo l'idrico.
fig.add_trace(
    go.Bar(
        x=df_macroaree["Area"],
        y=df_macroaree["Idrico"],
        name="Idrico",
        marker_color=colore_idrico,
        hovertemplate="Area: %{x}<br>Idrico: %{y:.2f} TWh<extra></extra>"
    )
)


# Aggiungo totale e quota sopra ogni barra.
fig.add_trace(
    go.Scatter(
        x=df_macroaree["Area"],
        y=df_macroaree["Totale_Rinnovabile"],
        mode="text",
        text=df_macroaree["Etichetta"],
        textposition="top center",
        showlegend=False,
        hoverinfo="skip"
    )
)


# Sistemo il layout.
fig.update_layout(
    title=f"Produzione rinnovabile per macroarea italiana - {anno_studio}<br><sup>Totale nazionale: {totale_nazionale:.1f} TWh</sup>",
    xaxis_title="Macroarea",
    yaxis_title="Produzione rinnovabile (TWh)",
    barmode="stack",
    height=650,
    title_x=0.5,
    legend_title="Fonte",
    hovermode="x unified"
)

fig.update_yaxes(showgrid=True)


# Mostro il grafico.
mostra_grafico(fig)

In [0]:
# Studio 2B - Mix percentuale rinnovabile per macroarea
#
# Questa seconda analisi non guarda più solo quanto produce ogni macroarea,
# ma come è composto il suo mix rinnovabile.
#
# Per ogni area viene calcolato il peso percentuale di:
# - Solare
# - Eolico
# - Idrico
#
# L'obiettivo è capire se una macroarea dipende soprattutto da una fonte
# oppure se presenta un mix più distribuito tra più tecnologie.
#
# Possibile utilizzo:
# questa vista può supportare analisi di diversificazione energetica,
# confronto tra modelli territoriali e valutazioni sulla dipendenza da specifiche fonti.
#
# In ambito gestionale o strategico può aiutare a distinguere aree più bilanciate
# da aree più esposte a una singola componente del mix rinnovabile.


# Copio il dataframe dello Studio 2A.
# Uso la stessa base dati perché qui cambio solo il punto di vista:
# dal valore assoluto alla composizione percentuale.
df_mix_area = df_macroaree.copy()


# Calcolo il peso percentuale del Solare dentro ogni macroarea.
df_mix_area["Solare_%"] = (
    df_mix_area["Solare"] / df_mix_area["Totale_Rinnovabile"] * 100
).round(1)


# Calcolo il peso percentuale dell'Eolico dentro ogni macroarea.
df_mix_area["Eolico_%"] = (
    df_mix_area["Eolico"] / df_mix_area["Totale_Rinnovabile"] * 100
).round(1)


# Calcolo il peso percentuale dell'Idrico dentro ogni macroarea.
df_mix_area["Idrico_%"] = (
    df_mix_area["Idrico"] / df_mix_area["Totale_Rinnovabile"] * 100
).round(1)


# Controllo i dati preparati.
display(
    df_mix_area[
        [
            "Area",
            "Solare_%",
            "Eolico_%",
            "Idrico_%"
        ]
    ]
)


# Creo il grafico 100% stacked.
fig = go.Figure()


# Aggiungo la quota solare.
fig.add_trace(
    go.Bar(
        x=df_mix_area["Area"],
        y=df_mix_area["Solare_%"],
        name="Solare",
        marker_color=colore_solare,
        text=df_mix_area["Solare_%"].astype(str) + "%",
        textposition="inside",
        hovertemplate="Area: %{x}<br>Solare: %{y:.1f}%<extra></extra>"
    )
)


# Aggiungo la quota eolica.
fig.add_trace(
    go.Bar(
        x=df_mix_area["Area"],
        y=df_mix_area["Eolico_%"],
        name="Eolico",
        marker_color=colore_eolico,
        text=df_mix_area["Eolico_%"].astype(str) + "%",
        textposition="inside",
        hovertemplate="Area: %{x}<br>Eolico: %{y:.1f}%<extra></extra>"
    )
)


# Aggiungo la quota idrica.
fig.add_trace(
    go.Bar(
        x=df_mix_area["Area"],
        y=df_mix_area["Idrico_%"],
        name="Idrico",
        marker_color=colore_idrico,
        text=df_mix_area["Idrico_%"].astype(str) + "%",
        textposition="inside",
        hovertemplate="Area: %{x}<br>Idrico: %{y:.1f}%<extra></extra>"
    )
)


# Sistemo il layout.
fig.update_layout(
    title=f"Composizione percentuale del mix rinnovabile per macroarea - {anno_studio}",
    xaxis_title="Macroarea",
    yaxis_title="Quota sul totale dell'area",
    barmode="stack",
    height=650,
    title_x=0.5,
    legend_title="Fonte",
    hovermode="x unified"
)

fig.update_yaxes(
    range=[0, 100],
    ticksuffix="%"
)


# Mostro il grafico.
mostra_grafico(fig)